In [1]:
import os
import torch
from torch.utils.data import Dataset, DataLoader
import random
from PIL import Image
import numpy as np
from sklearn.model_selection import train_test_split
import albumentations as A
from albumentations.pytorch import ToTensorV2
import glob
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import pytorch_lightning as pl
import torchmetrics
from pytorch_lightning.callbacks import EarlyStopping, ModelCheckpoint
import cv2

set Pathes

In [2]:
target_path = 'my_face_dataset/*.jpg'
all_targets = glob.glob(target_path)

control_path = 'img_align_celeba/*.jpg'
all_controls = glob.glob(control_path)

train-val-test split

In [3]:
train_target, temp_target = train_test_split(all_targets, test_size=0.2, random_state=42)
val_target, test_target = train_test_split(temp_target, test_size=0.5, random_state=42)

train_control, temp_control = train_test_split(all_controls, test_size=0.2, random_state=42)
val_control, test_control  = train_test_split(temp_control, test_size=0.5, random_state=42)

create custom Dataset

In [4]:
class MyDataset(Dataset):
    def __init__(self, target, control, transform=None):
        self.target = target
        self.control = control
        self.transform = transform

    def __len__(self):
        return len(self.target)
    
    def __getitem__(self, idx):
        anchor = self.target[idx]
        positive = random.choice(self.target)
        while positive == anchor:
            positive = random.choice(self.target)
        control = random.choice(self.control)

        anchor_img = self.load_image(anchor)
        positive_img = self.load_image(positive)
        control_img = self.load_image(control)

        if self.transform:
            anchor_img = self.transform(image=anchor_img)['image']
            positive_img = self.transform(image=positive_img)['image']
            control_img = self.transform(image=control_img)['image']

        return anchor_img, positive_img, control_img
    
    def load_image(self, path):
        img = Image.open(path).convert('RGB')
        return np.array(img).astype(np.float32) / 255.0

define Transformations

In [5]:
train_transform = A.Compose([
    A.Resize(height=128, width=128),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.Rotate(limit=15, p=0.3),
    A.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1, p=0.8), # new
    A.ShiftScaleRotate(
        shift_limit=0.05,
        scale_limit=0.1,
        rotate_limit=15,
        p=0.3
    ),
    A.Normalize(mean=[0.5063, 0.4258, 0.3832], std=[0.3090, 0.2858, 0.2865]),
    ToTensorV2()
])

val_transform = A.Compose([
    A.Resize(height=128, width=128),
    A.Normalize(mean=[0.5063, 0.4258, 0.3832], std=[0.3090, 0.2858, 0.2865]),
    ToTensorV2()
])

/home/nikita/.cnn/lib/python3.13/site-packages/albumentations/core/validation.py:114: UserWarning: ShiftScaleRotate is a special case of Affine transform. Please use Affine transform instead.
  original_init(self, **validated_kwargs)


datasets Initialization

In [6]:
train_dataset = MyDataset(target=train_target, control=train_control, transform=train_transform)

val_dataset = MyDataset(target=val_target, control=val_control, transform=val_transform)

test_dataset = MyDataset(target=test_target, control=test_control, transform=val_transform)

initialize Model

In [ ]:
class MyCNN(nn.Module):
    def __init__(self):
        super().__init__()
        
        self.block1 = nn.Sequential(nn.Conv2d(in_channels=3, out_channels=16, kernel_size=3, padding=1),
                                    nn.ReLU(),
                                    nn.MaxPool2d(2, 2))
        self.block2 = nn.Sequential(nn.Conv2d(in_channels=16, out_channels=32, kernel_size=3, padding=1),
                                    nn.ReLU(),
                                    nn.MaxPool2d(2, 2))
        self.block3 = nn.Sequential(nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3, padding=1),
                                    nn.ReLU(),
                                    nn.MaxPool2d(2, 2))
        self.block4 = nn.Sequential(nn.Conv2d(in_channels=64, out_channels=64, kernel_size=3, padding=1),
                                    nn.ReLU(),
                                    nn.MaxPool2d(2, 2))
        
        self.head = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 8 * 8, 512),
            nn.ReLU(),
            nn.BatchNorm1d(512),
            nn.Dropout(0.2),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.BatchNorm1d(256),
            nn.Dropout(0.1),
            nn.Linear(256, 128)
        )

    def forward(self, x):
        x = self.block1(x)
        x = self.block2(x)
        x = self.block3(x)
        x = self.block4(x)
        x = self.head(x)

        return x

In [ ]:
class FaceRecognizerSystem(pl.LightningModule):
    def __init__(self, lr=1e-4, margin=1.0, threshold=0.5):
        super().__init__()
        self.save_hyperparameters()

        self.model = MyCNN()

        self.criterion = nn.TripletMarginLoss(margin=self.hparams.margin, p=2)

        self.train_acc = torchmetrics.Accuracy(task='binary')
        self.val_acc = torchmetrics.Accuracy(task='binary')
        self.test_acc = torchmetrics.Accuracy(task='binary')

    def forward(self, x):
        embeddings = self.model(x)
        embeddings = F.normalize(embeddings, p=2, dim=1)
        return embeddings
    
    def training_step(self, batch, batch_idx):
            anchor, positive, control = batch

            emb_a = self(anchor)
            emb_p = self(positive)
            emb_c = self(control)

            loss = self.criterion(emb_a, emb_p, emb_c)

            dist_pos = F.pairwise_distance(emb_a, emb_p, p=2)
            dist_neg = F.pairwise_distance(emb_a, emb_c, p=2)

            preds_pos = (dist_pos < self.hparams.threshold).long()
            preds_neg = (dist_neg < self.hparams.threshold).long()
            
            preds = torch.cat([preds_pos, preds_neg])
            target = torch.cat([torch.ones_like(preds_pos), torch.zeros_like(preds_neg)])

            acc = self.train_acc(preds, target)

            self.log('train_loss', loss, prog_bar=True)
            self.log('train_acc', acc, prog_bar=True)
            self.log('dist_positive', dist_pos.mean(), prog_bar=False)
            self.log('dist_negative', dist_neg.mean(), prog_bar=False)

            return loss
    
    def validation_step(self, batch, batch_idx):
        anchor, positive, control = batch

        emb_a = self(anchor)
        emb_p = self(positive)
        emb_c = self(control)

        loss = self.criterion(emb_a, emb_p, emb_c)

        dist_pos = F.pairwise_distance(emb_a, emb_p, p=2)
        dist_neg = F.pairwise_distance(emb_a, emb_c, p=2)

        preds_pos = (dist_pos < self.hparams.threshold).long()
        preds_neg = (dist_neg < self.hparams.threshold).long()
        
        preds = torch.cat([preds_pos, preds_neg])
        target = torch.cat([torch.ones_like(preds_pos), torch.zeros_like(preds_neg)])

        acc = self.val_acc(preds, target)

        self.log('val_loss', loss, prog_bar=True)
        self.log('val_acc', acc, prog_bar=True)
        self.log('dist_positive', dist_pos.mean(), prog_bar=False)
        self.log('dist_negative', dist_neg.mean(), prog_bar=False)
        self.log('val_dist_pos', dist_pos.mean(), prog_bar=False)
        self.log('val_dist_neg', dist_neg.mean(), prog_bar=False)

        return loss
    
    def test_step(self, batch, batch_idx):
        anchor, positive, negative = batch
        
        emb_a = self(anchor)
        emb_p = self(positive)
        emb_c = self(negative)

        loss = self.criterion(emb_a, emb_p, emb_c)
        
        dist_pos = F.pairwise_distance(emb_a, emb_p, p=2)
        dist_neg = F.pairwise_distance(emb_a, emb_c, p=2)
        
        current_threshold = self.hparams.get('threshold', 0.5)
        preds_pos = (dist_pos < current_threshold).long()
        preds_neg = (dist_neg < current_threshold).long()
        
        preds = torch.cat([preds_pos, preds_neg])
        target = torch.cat([torch.ones_like(preds_pos), torch.zeros_like(preds_neg)])
        
        acc = self.test_acc(preds, target)

        self.log('test_loss', loss)
        self.log('test_acc', acc)
        self.log('test_dist_pos', dist_pos.mean())
        self.log('test_dist_neg', dist_neg.mean())
        
        return loss

    def configure_optimizers(self):
        optimizer = optim.Adam(self.parameters(), lr=self.hparams.lr)
        return optimizer

create Dataloader

In [9]:
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=4)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=4)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=4)

In [ ]:
early_stop_callback = EarlyStopping(
    monitor='val_loss',
    min_delta=0.001,
    patience=23,
    verbose=True,
    mode='min'
)

checkpoint_callback = ModelCheckpoint(
    monitor='val_acc',
    dirpath='my_models/',
    filename='face-net-{epoch:02d}-{val_acc:.3f}',
    save_top_k=1,
    mode='max',
    save_weights_only=True
)

In [ ]:
model = FaceRecognizerSystem()

trainer = pl.Trainer(
    max_epochs=150,
    callbacks=[early_stop_callback, checkpoint_callback],
    accelerator='auto',
    log_every_n_steps=5
)

trainer.fit(model, train_dataloaders=train_loader, val_dataloaders=val_loader)

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


model Test

In [ ]:
trainer.test(model, test_loader, ckpt_path='my_models/nice.ckpt')

lets see how it works!

In [ ]:
MODEL_PATH = "my_models/nice.ckpt"
REFERENCE_PHOTO_PATH = "my_face_dataset/cropped_Photo from 2026-04-30 15-55-45.211371.jpeg.jpg"
THRESHOLD = 0.4
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')


model = FaceRecognizerSystem.load_from_checkpoint(MODEL_PATH)
model.eval()
model.to(DEVICE)

transform = A.Compose([
    A.Resize(128, 128),
    A.Normalize(mean=(0.5, 0.5, 0.5), std=(0.5, 0.5, 0.5)),
    ToTensorV2()
])

ref_img = cv2.imread(REFERENCE_PHOTO_PATH)
ref_img = cv2.cvtColor(ref_img, cv2.COLOR_BGR2RGB)

ref_tensor = transform(image=ref_img)['image'].unsqueeze(0).to(DEVICE)

with torch.no_grad():
    reference_embedding = model(ref_tensor)

face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')

cap = cv2.VideoCapture(0)

while True:
    ret, frame = cap.read()
    if not ret:
        break

    gray_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

    faces = face_cascade.detectMultiScale(gray_frame, scaleFactor=1.1, minNeighbors=5, minSize=(60, 60))

    for (x, y, w, h) in faces:
        face_crop = rgb_frame[y:y+h, x:x+w]

        face_tensor = transform(image=face_crop)['image'].unsqueeze(0).to(DEVICE)

        with torch.no_grad():
            current_embedding = model(face_tensor)
            distance = F.pairwise_distance(reference_embedding, current_embedding, p=2).item()

        if distance < THRESHOLD:
            text = "Nikita"
            color = (0, 255, 0)
        else:
            text = "Unknown"
            color = (0, 0, 255)

        cv2.rectangle(frame, (x, y), (x+w, y+h), color, 2)
        cv2.putText(frame, text, (x, y-10), cv2.FONT_HERSHEY_SIMPLEX, 0.8, color, 2)

    cv2.imshow("CNN System", frame)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()